In [ ]:
# %% [markdown]
# # LLM Benchmark Arena — Extended Notebook
# This notebook runs a full model tournament: it writes questions, filters the weak ones,
# collects reference answers, asks every contestant to answer (twice each), has Grok judge
# the answers blindly three times, lets the losers revise, and prints a final leaderboard.
# Every stage is one cell, so you can run them one at a time and inspect the intermediate
# results. Tip: if any line below looks mysterious, ask the Cursor Agent to explain it.
#
# Note for plain-Python tooling: this is a NOTEBOOK file, and the fan-out cells use
# top-level `await` on purpose (Jupyter already runs an event loop, so `asyncio.run`
# would fail). Because of that, `python -m py_compile` reports those await lines;
# validate instead with:
#   python -c "import ast,pathlib; compile(pathlib.Path('llm_arena_extended.py').read_text(),'nb','exec',flags=ast.PyCF_ALLOW_TOP_LEVEL_AWAIT)"
# which is exactly how Jupyter itself compiles cell code.
# =============================================================================
# Stage 0: Setup — imports, environment keys, and one client per provider
# =============================================================================
# Bring in the standard toolbox plus the OpenAI SDK we use to talk to every provider.
import asyncio  # lets us run many model calls at the same time instead of one-by-one
import json  # reads and writes the structured text (JSON) that models send back
import os  # reads secret settings (API keys) from the environment
import random  # shuffles answers so the judge cannot tell who wrote what
import re  # pattern-matching helper used when a model wraps JSON in extra text
import statistics  # computes averages and spread (standard deviation) of scores
import time  # stops a stopwatch around each call so we can measure latency

In [ ]:
# requests is the simple blocking HTTP tool we use in Stage 4b to talk to
# Ollama's own management API (pulling and deleting models) — the OpenAI SDK
# cannot do those housekeeping calls, only chat.
import requests  # plain HTTP calls for Ollama's pull/delete/tags endpoints
from datetime import datetime, timezone  # timestamps for the log file
from pathlib import Path  # friendly way to point at files on disk

In [ ]:
# Load the .env file so the API keys sitting in it become environment variables.
from dotenv import load_dotenv  # the tool that reads .env files
from openai import AsyncOpenAI, OpenAI  # sync and async doorbells for every provider

In [ ]:
# IPython's display makes tables look pretty inside Jupyter; safe to skip elsewhere.
from IPython.display import display  # prettier printing of dataframes in the notebook

In [ ]:
# Read the .env file next to this notebook; override=True means fresh values win
# over anything already loaded, so re-running the cell picks up edits.
load_dotenv(override=True)  # load (or reload) the secrets from the .env file

In [ ]:
# Pull each provider's API key out of the environment (None if it is missing).
openai_api_key = os.getenv("OPENAI_API_KEY")  # key for OpenAI models
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")  # key for Claude models
google_api_key = os.getenv("GOOGLE_API_KEY")  # key for Gemini models
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")  # key for DeepSeek models
groq_api_key = os.getenv("GROQ_API_KEY")  # key for Groq-hosted models
grok_api_key = os.getenv("GROK_API_KEY")  # key for xAI's Grok (the judge)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")  # key for OpenRouter models

In [ ]:
# A tiny helper that prints only the first few characters of a key, so we can
# confirm a key was loaded without ever revealing the whole secret.
def _peek(key):  # given a key string (or None), show a safe preview
    # If the key is missing entirely, say so clearly instead of crashing later.
    if not key:  # the environment variable was not set
        return "MISSING"  # make the gap obvious in the debug printout
    # Otherwise show the first 6 characters followed by dots, like "sk-ab1...".
    return key[:6] + "..."  # enough to recognise the key, not enough to steal it

In [ ]:
# Print a one-line health check per provider, exactly like the first notebook does.
print("OpenAI key:     ", _peek(openai_api_key))  # confirm the OpenAI key loaded
print("Anthropic key:  ", _peek(anthropic_api_key))  # confirm the Anthropic key loaded
print("Google key:     ", _peek(google_api_key))  # confirm the Google key loaded
print("DeepSeek key:   ", _peek(deepseek_api_key))  # confirm the DeepSeek key loaded
print("Groq key:       ", _peek(groq_api_key))  # confirm the Groq key loaded
print("Grok key:       ", _peek(grok_api_key))  # confirm the Grok key loaded
print("OpenRouter key: ", _peek(openrouter_api_key))  # confirm the OpenRouter key loaded

In [ ]:
# Each provider's base URL can be overridden with an environment variable; the
# second argument to os.getenv is the default used when the variable is absent,
# so nothing changes unless you deliberately set an override.
anthropic_base_url = os.getenv("ANTHROPIC_BASE_URL", "https://api.anthropic.com/v1/")  # Claude's endpoint
gemini_base_url = os.getenv("GEMINI_BASE_URL", "https://generativelanguage.googleapis.com/v1beta/openai/")  # Gemini's OpenAI-compatible endpoint
deepseek_base_url = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com")  # DeepSeek's endpoint
groq_base_url = os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1")  # Groq's endpoint
grok_base_url = os.getenv("GROK_BASE_URL", "https://api.x.ai/v1")  # xAI Grok's endpoint (the judge)
openrouter_base_url = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")  # OpenRouter's endpoint
ollama_base_url = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")  # local Ollama's endpoint

In [ ]:
# Build one synchronous (regular, one-call-at-a-time) client per provider. Every
# provider speaks the OpenAI protocol, so they all use the same OpenAI class with
# a different base_url — exactly the style of the first notebook.
openai_client = OpenAI(api_key=openai_api_key)  # OpenAI uses its default URL
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_base_url)  # Claude's endpoint
gemini = OpenAI(api_key=google_api_key, base_url=gemini_base_url)  # Gemini's OpenAI-compatible endpoint
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_base_url)  # DeepSeek's endpoint
groq = OpenAI(api_key=groq_api_key, base_url=groq_base_url)  # Groq's endpoint
grok = OpenAI(api_key=grok_api_key, base_url=grok_base_url)  # xAI Grok's endpoint (the judge)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_base_url)  # OpenRouter's endpoint
ollama = OpenAI(base_url=ollama_base_url, api_key="ollama")  # local Ollama needs no real key

In [ ]:
# A name -> client lookup table, so the contestants list below can just say
# "groq" instead of repeating the whole client object.
CLIENTS = {  # one entry per provider we might send a contestant to
    "openai": openai_client,  # OpenAI models
    "anthropic": anthropic,  # Claude models
    "gemini": gemini,  # Gemini models
    "deepseek": deepseek,  # DeepSeek models
    "groq": groq,  # Groq-hosted open models
    "grok": grok,  # xAI Grok (reserved for judging)
    "openrouter": openrouter,  # OpenRouter-hosted models
    "ollama": ollama,  # local models via Ollama
}

In [ ]:
# Build the matching asynchronous clients — these let us fire off many calls at
# once in Stage 4 instead of waiting for each model to finish before the next.
async_openai_client = AsyncOpenAI(api_key=openai_api_key)  # async OpenAI doorbell
async_anthropic = AsyncOpenAI(api_key=anthropic_api_key, base_url=anthropic_base_url)  # async Claude doorbell
async_gemini = AsyncOpenAI(api_key=google_api_key, base_url=gemini_base_url)  # async Gemini doorbell
async_deepseek = AsyncOpenAI(api_key=deepseek_api_key, base_url=deepseek_base_url)  # async DeepSeek doorbell
async_groq = AsyncOpenAI(api_key=groq_api_key, base_url=groq_base_url)  # async Groq doorbell
async_grok = AsyncOpenAI(api_key=grok_api_key, base_url=grok_base_url)  # async Grok doorbell (for judging)
async_openrouter = AsyncOpenAI(api_key=openrouter_api_key, base_url=openrouter_base_url)  # async OpenRouter doorbell
async_ollama = AsyncOpenAI(base_url=ollama_base_url, api_key="ollama")  # async local Ollama doorbell

In [ ]:
# The async version of the name -> client lookup table.
ASYNC_CLIENTS = {  # mirrors CLIENTS but for the parallel fan-out
    "openai": async_openai_client,  # async OpenAI
    "anthropic": async_anthropic,  # async Claude
    "gemini": async_gemini,  # async Gemini
    "deepseek": async_deepseek,  # async DeepSeek
    "groq": async_groq,  # async Groq
    "grok": async_grok,  # async Grok (judge)
    "openrouter": async_openrouter,  # async OpenRouter
    "ollama": async_ollama,  # async local Ollama
}

In [ ]:
# pandas is optional: if it is installed we get pretty leaderboard tables, and if
# not we fall back to plain printed rows — so import it inside a safety net.
try:  # attempt the optional import
    import pandas as pd  # nice tables for the leaderboard

    HAVE_PANDAS = True  # remember that pretty tables are available
except ImportError:  # pandas is not installed in this environment
    HAVE_PANDAS = False  # remember to use plain prints instead

In [ ]:
# %%
# =============================================================================
# Shared helpers — JSON extraction, logging, and file locations
# =============================================================================
# Where the two output files live: right next to this notebook.
LOG_PATH = Path("arena_log.jsonl")  # one JSON line per call or judge pass (the diary)
RESULTS_PATH = Path("arena_results.json")  # the final leaderboard snapshot

In [ ]:
# Models love to wrap JSON in markdown fences or add chatter around it. This
# helper digs the real JSON object or array out of whatever text came back.
def extract_json(text):  # raw model text in, parsed Python object out
    # Remove ```json ... ``` style code fences that often surround the payload.
    cleaned = re.sub(r"```(?:json)?", "", text)  # strip opening/closing fences
    # Trim blank space from the ends so position searches behave.
    cleaned = cleaned.strip()  # drop leading/trailing whitespace
    # First try: maybe the whole thing is already clean JSON.
    try:  # optimistic parse of the cleaned text
        return json.loads(cleaned)  # return the parsed list or dict
    except json.JSONDecodeError:  # the text still has extra prose around the JSON
        pass  # fall through to the surgical approach below
    # Second try: slice from the first "[" or "{" to its matching last "]" or "}".
    openers = [i for i in (cleaned.find("["), cleaned.find("{")) if i != -1]  # where arrays/objects start
    closers = [cleaned.rfind("]"), cleaned.rfind("}")]  # where arrays/objects end
    # If we found both an opener and a closer after it, parse just that slice.
    if openers and max(closers) > min(openers):  # a plausible JSON island exists
        snippet = cleaned[min(openers) : max(closers) + 1]  # cut out just the JSON part
        return json.loads(snippet)  # parse the slice (may still raise — caller handles it)
    # Give up loudly so the caller can decide to retry or record a failure.
    raise ValueError("No JSON found in model output")  # nothing parseable was found

In [ ]:
# Append one JSON line to the diary file. Each call gets its own line so the log
# stays readable even if the notebook is interrupted halfway through.
def log_event(event):  # a dict describing one thing that happened
    # Stamp the event with the exact UTC moment it is being written down.
    event["logged_at"] = datetime.now(timezone.utc).isoformat()  # when this was recorded
    # Open the diary in append mode, write one line, and close it again.
    with LOG_PATH.open("a", encoding="utf-8") as f:  # "a" = add to the end, never erase
        f.write(json.dumps(event, ensure_ascii=False) + "\n")  # one JSON object per line

In [ ]:
# %%
# =============================================================================
# The contestants — ONE easily-editable list
# =============================================================================
# Each entry says who is playing ("name" = the exact model id), which provider
# doorbell to ring ("client" = a key into CLIENTS / ASYNC_CLIENTS above), and any
# extra settings to pass along ("kwargs"). To add or remove a model, edit one line.

CONTESTANTS = [  # the full tournament roster
    {"name": "gpt-5.4-nano", "client": "openai", "kwargs": {"reasoning_effort": "none"}},  # OpenAI's speedy small model, thinking switched off
    {"name": "claude-sonnet-4-6", "client": "anthropic", "kwargs": {}},  # Anthropic's Claude
    {"name": "gemini-3.1-flash-lite", "client": "gemini", "kwargs": {}},  # Google's lightweight Gemini
    {"name": "deepseek-v4-flash", "client": "deepseek", "kwargs": {}},  # DeepSeek's fast model
    {"name": "openai/gpt-oss-120b", "client": "groq", "kwargs": {}},  # big open model hosted on Groq
    {"name": "moonshotai/kimi-k2.6", "client": "openrouter", "kwargs": {}},  # Kimi via OpenRouter
    # NOTE for the local (ollama) entries below: your C drive only has room for
    # ONE downloaded model at a time, so Stage 4b plays them as a relay race —
    # pull one, run it, delete it, then the next. You can reorder the ollama
    # entries to control which model lives on your disk at which time.
    {"name": "llama3.2:1b", "client": "ollama", "kwargs": {}},  # tiny local Llama
    {"name": "gpt-oss:20b", "client": "ollama", "kwargs": {}},  # medium local open model
    {"name": "gemma4:latest", "client": "ollama", "kwargs": {}},  # local Gemma
    {"name": "qwen/qwen3-32b", "client": "groq", "kwargs": {}},  # NEW CONTESTANT — Qwen 3 hosted on Groq
    {"name": "meta-llama/llama-3.3-70b-versatile", "client": "groq", "kwargs": {}},  # NEW CONTESTANT — Llama 3.3 70B hosted on Groq
    {"name": "mistralai/mistral-large-2411", "client": "openrouter", "kwargs": {}},  # NEW CONTESTANT — Mistral Large via OpenRouter
]

In [ ]:
# A quick sanity print so we can see the roster before spending any API calls.
print(f"{len(CONTESTANTS)} contestants entered:")  # how many models are competing
for c in CONTESTANTS:  # walk through the roster one contestant at a time
    print(f"  - {c['name']}  (via {c['client']})")  # name and provider for each

In [ ]:
# %%
# =============================================================================
# Stage 1: Question generation (orchestrator)
# =============================================================================
# gpt-6-luna invents the quiz questions. We ask for strict JSON so the rest of
# the pipeline can read the questions without any human copy-pasting.

N_QUESTIONS = 6  # how many questions we want for the tournament

# The six categories keep the quiz varied — one question per category.
CATEGORIES = [  # the flavours of hard question we want covered
    "ethics",  # right vs wrong dilemmas
    "epistemics",  # how we know what we know
    "ambiguity",  # questions with no single clean answer
    "second-order effects",  # consequences of consequences
    "practical wisdom",  # judgement calls from real life
    "self-reference",  # questions that bend back on themselves
]

In [ ]:
# The instruction we send to the question-writer model.
question_prompt = f"""You are writing {N_QUESTIONS} hard interview questions to benchmark large language models.

Cover exactly these categories, one question each: {", ".join(CATEGORIES)}.

Each question should reward insight, accuracy, brevity, and nuance — not trivia recall.

Return ONLY a JSON array of objects, each with keys "category" and "question". No markdown fences, no commentary."""  # strict JSON out, nothing else

In [ ]:
# Ask the orchestrator model for the questions (single blocking call is fine
# here). If the reply is not parseable JSON, retry ONCE with a firmer "return
# ONLY valid JSON" message before giving up.
questions = None  # will hold the parsed list of question dicts
q_messages = [{"role": "user", "content": question_prompt}]  # the brief we send
for attempt in range(2):  # attempt 0 = normal, attempt 1 = sterner retry
    # Ask gpt-6-luna to write (or re-write) the quiz.
    q_response = openai_client.chat.completions.create(  # gpt-6-luna writes the quiz
        model="gpt-6-luna",  # the orchestrator model
        messages=q_messages,  # the current brief (sterner on the retry)
    )
    # Pull the raw text of the model's reply out of the response envelope.
    q_raw = q_response.choices[0].message.content  # the model's answer as plain text
    # Try to parse the JSON; on failure, loop around for the one retry.
    try:  # hopeful parse
        questions = extract_json(q_raw)  # a list of {"category": ..., "question": ...}
        break  # success — no retry needed
    except Exception:  # broken JSON
        # The retry brief: same request, plus an unmistakable formatting demand.
        q_messages = [{"role": "user", "content": question_prompt + "\n\nREMINDER: return ONLY valid JSON. No prose, no markdown fences."}]  # sterner brief
        print("Question generation returned unparseable JSON — retrying once with a sterner prompt.")  # warn the human

In [ ]:
# If even the retry failed, there is no quiz to run — stop here with a clear message.
if questions is None:  # both attempts failed to produce JSON
    raise RuntimeError("gpt-6-luna did not return parseable JSON for the questions, even after a retry.")  # fail loudly

In [ ]:
# Validate the shape before Stage 2: we need a list of dicts, each carrying
# both a "category" and a "question" key. Anything else is dropped with a note.
if not isinstance(questions, list):  # the model returned an object, not an array
    raise RuntimeError("Question generation did not return a JSON array of questions.")  # fail loudly
well_formed = [q for q in questions if isinstance(q, dict) and "category" in q and "question" in q]  # keep only valid entries
if len(well_formed) < len(questions):  # some entries were malformed
    print(f"Dropped {len(questions) - len(well_formed)} malformed question entries (missing 'category' or 'question').")  # warn the human
questions = well_formed  # continue with the valid ones only
if not questions:  # nothing usable survived validation
    raise RuntimeError("No valid questions survived shape validation — cannot run the tournament.")  # fail loudly

In [ ]:
# Give each question a short stable id (q1, q2, ...) so later stages can refer to it.
for i, q in enumerate(questions, start=1):  # number the questions starting at 1
    q["id"] = f"q{i}"  # attach the id to the question dict

In [ ]:
# Record the token usage of this call if the API told us (some providers omit it).
q_usage = q_response.usage.model_dump() if getattr(q_response, "usage", None) else None  # token counts when present

In [ ]:
# Write the question-generation call into the diary.
log_event({"stage": "generate_questions", "model": "gpt-6-luna", "usage": q_usage, "n_questions": len(questions)})  # diary entry

In [ ]:
# Show the generated questions so a human can veto before we spend more calls.
for q in questions:  # walk through the freshly written questions
    print(f"[{q['id']}] ({q['category']}) {q['question']}")  # id, category, and text

In [ ]:
# %%
# =============================================================================
# Stage 2: Question critic — Claude keeps only the strong questions
# =============================================================================
# Each question is graded by claude-sonnet-4-6 with {"keep": bool, "reason": ...}.
# Weak or broken questions get dropped before we spend money asking 12 models.

kept_questions = []  # questions that pass inspection will live here

In [ ]:
# Judge the questions one at a time — this is cheap and keeps reasons readable.
for q in questions:  # one critic call per question
    # The critic's brief: score this single question and answer with strict JSON.
    critic_prompt = f"""You are a ruthless quiz editor. Judge this candidate question:

Category: {q["category"]}
Question: {q["question"]}

Keep it only if it is answerable, non-trivial, and rewards insight/accuracy/brevity/nuance.

Return ONLY JSON: {{"keep": true or false, "reason": "one short sentence"}}."""  # strict JSON verdict

    # Ask Claude to grade the question.
    c_response = anthropic.chat.completions.create(  # claude-sonnet-4-6 plays critic
        model="claude-sonnet-4-6",  # the critic model
        messages=[{"role": "user", "content": critic_prompt}],  # the grading brief
    )

    # Get the critic's raw reply text.
    c_raw = c_response.choices[0].message.content  # what Claude said back

    # Try to parse the verdict; if the JSON is broken, keep the question but note it.
    try:  # the happy path — Claude returned clean JSON
        verdict = extract_json(c_raw)  # {"keep": ..., "reason": ...}
    except Exception:  # the reply was not parseable JSON
        verdict = {"keep": True, "reason": "critic output unparseable — kept by default"}  # fail open

    # Show the keep/drop decision as one row of a little table.
    print(f"[{q['id']}] keep={verdict.get('keep')} — {verdict.get('reason')}")  # the verdict row

    # Write the verdict to the diary.
    log_event({"stage": "critic", "model": "claude-sonnet-4-6", "question_id": q["id"], "verdict": verdict})  # diary entry

    # If the critic said keep, add the question (with its reason) to the kept pile.
    if verdict.get("keep"):  # the question survived
        q["critic_reason"] = verdict.get("reason", "")  # remember why it was kept
        kept_questions.append(q)  # promote it to the tournament list

# Announce how many questions survived the critic.
print(f"\n{len(kept_questions)} of {len(questions)} questions kept.")  # survival count

# Safety net: if the critic hated everything, keep the originals so the show goes on.
if not kept_questions:  # an empty quiz would break every later stage
    kept_questions = questions  # fall back to the full set
    print("Critic dropped everything — falling back to all questions.")  # warn the human

In [ ]:
# %%
# =============================================================================
# Stage 3: Reference answers — Claude writes the gold standard
# =============================================================================
# For every kept question, claude-sonnet-4-6 writes a short model answer plus
# 3–4 grading criteria. The judge later compares contestants against these.

reference_answers = {}  # question_id -> {"answer": ..., "criteria": [...]}

In [ ]:
# Write the references one at a time; this is sequential and cheap.
for q in kept_questions:  # one reference per kept question
    # The brief: answer the question well and list what a great answer must do.
    ref_prompt = f"""Write a short gold-standard answer to this question, then list 3–4 grading criteria
that a great answer must satisfy.

Question: {q["question"]}

Return ONLY JSON: {{"answer": "...", "criteria": ["...", "...", "..."]}}."""  # strict JSON out

    # Ask Claude for the reference answer and criteria.
    r_response = anthropic.chat.completions.create(  # claude-sonnet-4-6 writes the gold standard
        model="claude-sonnet-4-6",  # the reference writer
        messages=[{"role": "user", "content": ref_prompt}],  # the brief
    )

    # Get the raw reply text.
    r_raw = r_response.choices[0].message.content  # Claude's reference as text

    # Parse it; if parsing fails, store an empty skeleton so later stages survive.
    try:  # the happy path
        reference_answers[q["id"]] = extract_json(r_raw)  # store answer + criteria
    except Exception:  # unparseable reference
        reference_answers[q["id"]] = {"answer": "", "criteria": []}  # empty fallback
        print(f"[{q['id']}] reference unparseable — stored empty fallback")  # warn the human

    # Diary entry for this reference-answer call.
    log_event({"stage": "reference", "model": "claude-sonnet-4-6", "question_id": q["id"]})  # diary entry

    # Show a short preview so we can eyeball the gold standard.
    preview = reference_answers[q["id"]]["answer"][:120]  # first 120 characters
    print(f"[{q['id']}] reference: {preview}...")  # preview row

In [ ]:
# %%
# =============================================================================
# Stage 4: Async fan-out — every CLOUD contestant answers every question, twice
# =============================================================================
# ----- Stage 4: wide fan-out with per-provider speed limits (cloud only) -----

REPS = 2  # each contestant answers each question this many times (for variance)

In [ ]:
# Split the roster into two travel groups. WHY: your C drive is nearly full, so
# only ONE Ollama model can exist on disk at a time. The cloud models can all
# answer in parallel (they live on someone else's computer), but the local
# "ollama" contestants must take turns in a pull -> run -> delete relay race,
# which happens in the NEXT cell (Stage 4b). This cell therefore handles only
# the non-ollama contestants.
cloud_contestants = [c for c in CONTESTANTS if c["client"] != "ollama"]  # parallel group (this cell)
ollama_contestants = [c for c in CONTESTANTS if c["client"] == "ollama"]  # relay-race group (Stage 4b)

# Per-provider speed limits: at most this many simultaneous calls per provider.
# Ollama runs on the local machine, so we serialize it (1 at a time) to avoid
# overloading the GPU; the cloud providers comfortably handle 4 at once.
PROVIDER_LIMITS = {"ollama": 1}  # special-case: local models go one at a time (used by Stage 4b)
DEFAULT_LIMIT = 4  # every other provider gets 4 parallel slots

# Build one semaphore (a bouncer with a fixed number of entry tickets) per provider.
# We deliberately build a bouncer for EVERY provider in the roster — including
# "ollama", which this cell no longer uses. That looks unused here, but Stage 4b
# borrows the ollama bouncer (limit 1) so its local calls stay serialized, so
# keeping it is harmless and saves Stage 4b from building its own.
provider_names = {c["client"] for c in CONTESTANTS}  # all providers in the roster (incl. ollama)
SEMAPHORES = {  # provider -> its bouncer
    p: asyncio.Semaphore(PROVIDER_LIMITS.get(p, DEFAULT_LIMIT))  # 1 for ollama, 4 for the rest
    for p in provider_names  # one bouncer per provider in the roster
}


# One async job: a single contestant answering a single question on a single rep.
async def answer_one(contestant, question, rep):  # one question, one model, one try
    # Look up this contestant's async doorbell and its provider's bouncer.
    client = ASYNC_CLIENTS[contestant["client"]]  # which provider to call
    sem = SEMAPHORES[contestant["client"]]  # which bouncer to ask permission from

    # Build the chat messages: just the question, asked plainly.
    messages = [{"role": "user", "content": question["question"]}]  # the prompt

    # Wait for the bouncer to let us in, start the stopwatch, and make the call.
    async with sem:  # at most N calls per provider run at once
        t0 = time.perf_counter()  # start the latency stopwatch
        resp = await client.chat.completions.create(  # the actual model call
            model=contestant["name"],  # which model answers
            messages=messages,  # the question
            **contestant["kwargs"],  # any per-model extras (e.g. reasoning_effort)
        )
        latency = time.perf_counter() - t0  # stop the stopwatch

    # Extract the answer text and token usage (usage may be missing on some APIs).
    text = resp.choices[0].message.content  # the model's answer
    usage = resp.usage.model_dump() if getattr(resp, "usage", None) else None  # tokens when present

    # Write this call to the diary with everything we might want to analyse later.
    log_event({  # one diary line per call
        "stage": "answer",  # which pipeline stage produced this
        "model": contestant["name"],  # who answered
        "provider": contestant["client"],  # via which provider
        "question_id": question["id"],  # which question
        "rep": rep,  # which repetition
        "latency_s": round(latency, 3),  # how long it took
        "chars": len(text or ""),  # how long the answer is
        "usage": usage,  # token counts when available
    })

    # Hand back a tidy record of this answer for the judging stage.
    return {  # the answer record
        "model": contestant["name"],  # who wrote it
        "provider": contestant["client"],  # via which provider
        "question_id": question["id"],  # which question
        "rep": rep,  # which repetition
        "answer": text,  # the answer text
        "latency_s": latency,  # how long the call took
        "usage": usage,  # token counts when available
        "error": None,  # no error happened
    }


# Build the cloud work list: every CLOUD contestant x every kept question x
# every rep. The ollama contestants are intentionally absent — they run later,
# one at a time, in Stage 4b.
jobs = [  # the complete work list for the cloud fan-out
    answer_one(c, q, rep)  # one job
    for c in cloud_contestants  # for every cloud contestant (no ollama here)
    for q in kept_questions  # for every kept question
    for rep in range(REPS)  # repeated REPS times
]

# A parallel list saying WHO each job is — (model, question, rep) — in exactly
# the same order as `jobs`. asyncio.gather preserves order, so results[i]
# always belongs to job_meta[i]; this lets failure logs name the culprit.
job_meta = [  # one identity tag per job, same comprehension order as above
    (c["name"], q["id"], rep)  # who was asked, which question, which rep
    for c in cloud_contestants  # for every cloud contestant (no ollama here)
    for q in kept_questions  # for every kept question
    for rep in range(REPS)  # repeated REPS times
]

cloud_provider_names = {c["client"] for c in cloud_contestants}  # the cloud providers actually in play
print(f"Firing {len(jobs)} cloud model calls across {len(cloud_provider_names)} providers...")  # how much work

# asyncio.gather with return_exceptions=True runs everything at once and turns
# individual crashes into returned Exception objects instead of killing the run —
# one broken model must never take down the whole tournament.
#
# Why bare `await` instead of asyncio.run()? Jupyter already runs an event loop
# under the hood, so calling asyncio.run() would error with "event loop already
# running"; a top-level await simply borrows that existing loop.
results = await asyncio.gather(*jobs, return_exceptions=True)  # run all jobs concurrently

# Split the results into successful answers and recorded failures. Zipping with
# job_meta is safe because gather returns results in the same order as the jobs.
answers = []  # successful answers go here
failures = []  # failures go here, recorded but never fatal
for (model_name, question_id, rep), res in zip(job_meta, results):  # walk results with their identity tags
    if isinstance(res, Exception):  # this job crashed
        failures.append(f"{model_name} / {question_id} / rep {rep}: {res}")  # record the error with its identity
        log_event({"stage": "answer", "model": model_name, "question_id": question_id, "rep": rep, "error": str(res)})  # diary entry naming the culprit
    else:  # this job succeeded
        answers.append(res)  # keep the answer record

# Report how the cloud fan-out went. The local (ollama) answers are still
# missing at this point — Stage 4b below adds them to these very same lists.
print(f"{len(answers)} cloud answers collected, {len(failures)} failures (recorded, not fatal).")  # summary so far

In [ ]:
# %% 
# MANUAL Ollama round — the model named below must ALREADY be installed.
CURRENT_OLLAMA_MODEL = "gemma4:latest"  # <-- the only line you change between rounds

# Find this model's entry in the roster (its name, provider, extra settings).
contestant = next(c for c in CONTESTANTS if c["name"] == CURRENT_OLLAMA_MODEL)  # look up the one contestant we want

# Build this model's full work list: every kept question, answered REPS times.
manual_jobs = [answer_one(contestant, q, rep) for q in kept_questions for rep in range(REPS)]  # one job per (question, rep)
manual_meta = [(q["id"], rep) for q in kept_questions for rep in range(REPS)]  # matching identity tags for failure logging

# Run the jobs (the ollama "bouncer" from Stage 4 keeps them one-at-a-time anyway).
manual_results = await asyncio.gather(*manual_jobs, return_exceptions=True)  # gather keeps going even if one call crashes

done = 0  # count how many answers we actually got
for (qid, rep), res in zip(manual_meta, manual_results):  # walk results together with their identity tags
    if isinstance(res, Exception):  # this particular call crashed
        failures.append(f"{CURRENT_OLLAMA_MODEL} / {qid} / rep {rep}: {res}")  # remember it, with full identity
        log_event({"stage": "answer", "model": CURRENT_OLLAMA_MODEL, "question_id": qid, "rep": rep, "error": str(res)})  # diary entry
    else:  # this call worked
        answers.append(res)  # lands in the SAME list as the cloud answers — judging can't tell the difference
        done += 1  # one more success
print(f"{CURRENT_OLLAMA_MODEL} answered {done}/{len(manual_jobs)} jobs")  # disk-friendly status line

In [ ]:
# %% 
# MANUAL Ollama round — the model named below must ALREADY be installed.
CURRENT_OLLAMA_MODEL = "llama3.2:1b"  # <-- the only line you change between rounds

# Find this model's entry in the roster (its name, provider, extra settings).
contestant = next(c for c in CONTESTANTS if c["name"] == CURRENT_OLLAMA_MODEL)  # look up the one contestant we want

# Build this model's full work list: every kept question, answered REPS times.
manual_jobs = [answer_one(contestant, q, rep) for q in kept_questions for rep in range(REPS)]  # one job per (question, rep)
manual_meta = [(q["id"], rep) for q in kept_questions for rep in range(REPS)]  # matching identity tags for failure logging

# Run the jobs (the ollama "bouncer" from Stage 4 keeps them one-at-a-time anyway).
manual_results = await asyncio.gather(*manual_jobs, return_exceptions=True)  # gather keeps going even if one call crashes

done = 0  # count how many answers we actually got
for (qid, rep), res in zip(manual_meta, manual_results):  # walk results together with their identity tags
    if isinstance(res, Exception):  # this particular call crashed
        failures.append(f"{CURRENT_OLLAMA_MODEL} / {qid} / rep {rep}: {res}")  # remember it, with full identity
        log_event({"stage": "answer", "model": CURRENT_OLLAMA_MODEL, "question_id": qid, "rep": rep, "error": str(res)})  # diary entry
    else:  # this call worked
        answers.append(res)  # lands in the SAME list as the cloud answers — judging can't tell the difference
        done += 1  # one more success
print(f"{CURRENT_OLLAMA_MODEL} answered {done}/{len(manual_jobs)} jobs")  # disk-friendly status line

In [ ]:
# %% 
# MANUAL Ollama round — the model named below must ALREADY be installed.
CURRENT_OLLAMA_MODEL = "gpt-oss:20b"  # <-- the only line you change between rounds

# Find this model's entry in the roster (its name, provider, extra settings).
contestant = next(c for c in CONTESTANTS if c["name"] == CURRENT_OLLAMA_MODEL)  # look up the one contestant we want

# Build this model's full work list: every kept question, answered REPS times.
manual_jobs = [answer_one(contestant, q, rep) for q in kept_questions for rep in range(REPS)]  # one job per (question, rep)
manual_meta = [(q["id"], rep) for q in kept_questions for rep in range(REPS)]  # matching identity tags for failure logging

# Run the jobs (the ollama "bouncer" from Stage 4 keeps them one-at-a-time anyway).
manual_results = await asyncio.gather(*manual_jobs, return_exceptions=True)  # gather keeps going even if one call crashes

done = 0  # count how many answers we actually got
for (qid, rep), res in zip(manual_meta, manual_results):  # walk results together with their identity tags
    if isinstance(res, Exception):  # this particular call crashed
        failures.append(f"{CURRENT_OLLAMA_MODEL} / {qid} / rep {rep}: {res}")  # remember it, with full identity
        log_event({"stage": "answer", "model": CURRENT_OLLAMA_MODEL, "question_id": qid, "rep": rep, "error": str(res)})  # diary entry
    else:  # this call worked
        answers.append(res)  # lands in the SAME list as the cloud answers — judging can't tell the difference
        done += 1  # one more success
print(f"{CURRENT_OLLAMA_MODEL} answered {done}/{len(manual_jobs)} jobs")  # disk-friendly status line

In [ ]:
# %%
# =============================================================================
# Stage 5: Blinded judging, 3 passes — Grok is ALWAYS the judge
# =============================================================================
# ----- Stage 5: Grok as judge, hardened -----
# For each question and each of 3 passes we shuffle the answers, relabel them
# A/B/C..., and ask grok-4.7 to score each on insight, accuracy, brevity and
# nuance (1–10) plus a short critique. Only the private label map knows who is
# who, so the judge cannot play favourites.

JUDGE_MODEL = "grok-4.7"  # the one and only judge
JUDGE_PASSES = 3  # how many independent shuffled passes per question
RUBRIC = ["insight", "accuracy", "brevity", "nuance"]  # the four scored dimensions
JUDGE_SEM = asyncio.Semaphore(4)  # at most 4 judging calls at once (Grok is a cloud API)


# Build the judge's prompt for one question: reference answer, rubric, and the
# blinded answers, demanding strict JSON back.
def build_judge_prompt(question, reference, labelled):  # labelled = [(label, answer_text), ...]
    # Turn the (label, answer) pairs into a readable block like "Answer A: ...".
    block = "\n\n".join(f"Answer {label}:\n{text}" for label, text in labelled)  # the blinded answers
    # Assemble the full judging brief.
    return f"""You are a strict, fair judge of LLM answers. You do not know which model wrote which answer.

Question: {question["question"]}

Reference (gold-standard) answer:
{reference.get("answer", "")}

Grading criteria from the reference writer:
{json.dumps(reference.get("criteria", []), indent=2)}

Score EACH answer below on these dimensions, each 1–10:
- insight (does it say something non-obvious?)
- accuracy (is it correct?)
- brevity (is it concise without losing substance?)
- nuance (does it handle ambiguity and trade-offs?)

{block}

Return ONLY valid JSON — an object mapping each label to its scores and a one-sentence critique:
{{"A": {{"insight": 7, "accuracy": 8, "brevity": 6, "nuance": 7, "critique": "..."}}, "B": {{...}}}}
No markdown fences, no commentary."""  # strict JSON out


# One async judging job: score one question's answers on one pass.
async def judge_pass(question, q_answers, pass_idx):  # one question, one shuffled pass
    # Copy the answers and shuffle them fresh — every pass gets a new order.
    shuffled = list(q_answers)  # copy so we never mutate the original list
    random.shuffle(shuffled)  # fresh shuffle per pass

    # Assign blind labels A, B, C... in shuffled order and build the private map.
    labels = [chr(ord("A") + i) for i in range(len(shuffled))]  # A, B, C, ...
    label_map = {  # label -> which model/rep actually wrote it (kept private)
        label: {"model": a["model"], "rep": a["rep"]}  # the de-anonymisation key
        for label, a in zip(labels, shuffled)  # pair each label with its answer
    }
    labelled = [(label, a["answer"]) for label, a in zip(labels, shuffled)]  # what the judge sees

    # Build the prompt with the reference answer and blinded answers.
    prompt = build_judge_prompt(question, reference_answers[question["id"]], labelled)  # the brief

    # Ask Grok to judge, retrying once with a sterner prompt if the JSON is broken.
    scores = None  # will hold the parsed scores
    for attempt in range(2):  # attempt 0 = normal, attempt 1 = sterner retry
        # On the retry, append a firm reminder to return ONLY valid JSON.
        this_prompt = prompt if attempt == 0 else prompt + "\n\nREMINDER: return ONLY valid JSON. No prose, no fences."  # sterner on retry
        # Make the judging call behind the judge bouncer, timing it.
        async with JUDGE_SEM:  # at most 4 judge calls at once
            t0 = time.perf_counter()  # start the stopwatch
            resp = await async_grok.chat.completions.create(  # grok-4.7 judges
                model=JUDGE_MODEL,  # always the same judge
                messages=[{"role": "user", "content": this_prompt}],  # the brief
            )
            latency = time.perf_counter() - t0  # stop the stopwatch
        # Grab the judge's raw reply text.
        raw = resp.choices[0].message.content  # what Grok said back
        # Try to parse the scores; on failure, loop around for the one retry.
        try:  # hopeful parse
            scores = extract_json(raw)  # {"A": {...}, "B": {...}, ...}
            break  # success — no retry needed
        except Exception:  # broken JSON
            if attempt == 1:  # the retry also failed
                log_event({"stage": "judge", "model": JUDGE_MODEL, "question_id": question["id"], "pass": pass_idx, "error": "unparseable judge output"})  # record and move on
                return []  # this pass yields no scores, but the run continues

    # Token usage of the judge call, when the API reports it.
    usage = resp.usage.model_dump() if getattr(resp, "usage", None) else None  # tokens when present

    # Write the judge pass to the diary (including the private label map).
    log_event({  # one diary line per judge pass
        "stage": "judge",  # which pipeline stage
        "model": JUDGE_MODEL,  # who judged
        "question_id": question["id"],  # which question
        "pass": pass_idx,  # which of the 3 passes
        "latency_s": round(latency, 3),  # how long judging took
        "usage": usage,  # tokens when available
        "label_map": label_map,  # private de-anonymisation map
        "scores": scores,  # the raw scores per label
    })

    # De-anonymise: turn label-keyed scores into answer-keyed score records.
    records = []  # one record per answer scored in this pass
    for label, scores_for_label in scores.items():  # walk the judge's JSON
        # Skip any label the judge invented that is not in our private map.
        if label not in label_map:  # hallucinated label
            continue  # ignore it
        who = label_map[label]  # which model/rep this label hides
        records.append({  # the de-anonymised score record
            "model": who["model"],  # who was actually scored
            "rep": who["rep"],  # which repetition of theirs
            "question_id": question["id"],  # which question
            "pass": pass_idx,  # which judging pass
            "scores": {k: scores_for_label.get(k) for k in RUBRIC},  # the four numbers
            "critique": scores_for_label.get("critique", ""),  # the judge's one-liner
        })
    return records  # hand back this pass's de-anonymised scores


# Group the answers by question so each judging job sees one question's answers.
by_question = {}  # question_id -> list of answer records
for a in answers:  # walk every collected answer
    by_question.setdefault(a["question_id"], []).append(a)  # file it under its question

# Build one judging job per kept question per pass. Questions that NO model
# answered successfully are skipped — the judge must never see an empty answer block.
judge_jobs = [  # the full judging work list
    judge_pass(q, by_question.get(q["id"], []), p)  # one job
    for q in kept_questions  # for every kept question
    if by_question.get(q["id"])  # skip questions with zero successful answers
    for p in range(JUDGE_PASSES)  # for each of the 3 passes
]

print(f"Judging: {len(judge_jobs)} judge calls ({len(kept_questions)} questions x {JUDGE_PASSES} passes)...")  # the workload

# Run all judge jobs concurrently; a broken judge call is recorded, never fatal.
judge_results = await asyncio.gather(*judge_jobs, return_exceptions=True)  # top-level await (Jupyter already has an event loop)

# Flatten the per-pass lists into one big score table, catching any crashes.
score_records = []  # every de-anonymised score record lands here
for res in judge_results:  # walk gather's output
    if isinstance(res, Exception):  # a judge job crashed outright
        log_event({"stage": "judge", "model": JUDGE_MODEL, "error": str(res)})  # record it
        print(f"Judge call failed (recorded): {res}")  # warn the human
    else:  # a list of score records from one pass
        score_records.extend(res)  # merge into the big table

print(f"Collected {len(score_records)} scored answers across all passes.")  # how much got scored

In [ ]:
# %%
# =============================================================================
# Stage 6: Evaluator-optimizer loop — the bottom half revises using the critique
# =============================================================================
# Grok's critique is handed back to the weaker half of the field; they revise,
# Grok re-judges the revisions blind, and we measure who actually improved.
# Capped at MAX_LOOPS rounds and stops early if a whole round helps nobody.

MAX_LOOPS = 2  # at most two rounds of revise-and-rejudge

# Re-judge scores from this stage live in their OWN list, completely separate
# from score_records. They are used ONLY for the before/after delta report in
# this cell — Stage 7's leaderboard, rep variance, and judge self-consistency
# must be computed from the original Stage 5 passes alone, because mixing
# original and revised answer texts under the same (model, question, rep) key
# would measure the revision instead of the judge's wobble.
revision_records = []  # re-judge scores (pass ids >= 100) land here, never in score_records


# Helper: mean total score (sum of the 4 rubric dimensions) per model, given a
# set of score records. Used to decide who is in the bottom half.
def mean_total_by_model(records):  # records -> {model: mean total score}
    totals = {}  # model -> list of per-record totals
    for r in records:  # walk every score record
        vals = [v for v in r["scores"].values() if isinstance(v, (int, float))]  # numeric scores only
        if vals:  # only count records that actually have numbers
            totals.setdefault(r["model"], []).append(sum(vals))  # this answer's total (out of 40)
    # Average each model's totals; models with no scores get 0.
    return {m: statistics.mean(ts) if ts else 0.0 for m, ts in totals.items()}  # model -> mean total


# Work out the current standings from all the Stage 5 scores.
current_totals = mean_total_by_model(score_records)  # model -> mean total so far

# Keep the best answer text we have per (model, question, rep) — revisions update it.
best_answers = {(a["model"], a["question_id"], a["rep"]): a for a in answers}  # start with originals

# Keep the latest critique per (model, question, rep) so revisions can use it.
latest_critique = {(r["model"], r["question_id"], r["rep"]): r["critique"] for r in score_records}  # from judging

# Run the revise-and-rejudge loop.
for loop_idx in range(1, MAX_LOOPS + 1):  # round 1, then round 2 at most
    # Rank the models by current mean total and pick the bottom half.
    ranked = sorted(current_totals.items(), key=lambda kv: kv[1])  # worst first
    bottom_models = {m for m, _ in ranked[: max(1, len(ranked) // 2)]}  # the weaker half (at least 1)
    print(f"\nRevision loop {loop_idx}: bottom half = {sorted(bottom_models)}")  # who is revising

    # One revision job: rewrite one answer given Grok's critique of it.
    async def revise_one(model, question, rep, original_answer, critique):  # one rewrite
        # Find the contestant entry so we know which provider doorbell to ring.
        contestant = next(c for c in CONTESTANTS if c["name"] == model)  # the model's roster entry
        client = ASYNC_CLIENTS[contestant["client"]]  # its async doorbell
        sem = SEMAPHORES[contestant["client"]]  # its provider bouncer
        # The revision brief: original answer, Grok's critique, and "keep it short".
        rev_prompt = f"""Question: {question["question"]}

Your previous answer:
{original_answer}

A judge gave this critique of your answer:
{critique}

Revise your answer given this feedback. Keep it short."""  # the rewrite brief
        # Make the revision call behind the provider bouncer, timing it.
        async with sem:  # respect the provider's speed limit
            t0 = time.perf_counter()  # start the stopwatch
            resp = await client.chat.completions.create(  # the revision call
                model=model,  # the model revises itself
                messages=[{"role": "user", "content": rev_prompt}],  # the brief
                **contestant["kwargs"],  # per-model extras
            )
            latency = time.perf_counter() - t0  # stop the stopwatch
        # Extract the revised text and usage.
        text = resp.choices[0].message.content  # the revised answer
        usage = resp.usage.model_dump() if getattr(resp, "usage", None) else None  # tokens when present
        # Diary entry for the revision call.
        log_event({"stage": "revise", "model": model, "question_id": question["id"], "rep": rep, "latency_s": round(latency, 3), "chars": len(text or ""), "usage": usage, "loop": loop_idx})  # diary
        # Hand back the revised answer record.
        return {"model": model, "provider": contestant["client"], "question_id": question["id"], "rep": rep, "answer": text, "latency_s": latency, "usage": usage, "error": None, "loop": loop_idx}  # the record

    # Build the revision jobs: every answer written by a bottom-half model.
    rev_jobs = []  # revision work list for this loop
    for (model, qid, rep), ans in best_answers.items():  # every answer we hold
        if model in bottom_models:  # only the bottom half revises
            question = next(q for q in kept_questions if q["id"] == qid)  # the question it answered
            critique = latest_critique.get((model, qid, rep), "No specific critique available.")  # Grok's feedback
            rev_jobs.append(revise_one(model, question, rep, ans["answer"], critique))  # queue the rewrite

    # Run all revisions concurrently; failures are recorded, never fatal.
    rev_results = await asyncio.gather(*rev_jobs, return_exceptions=True)  # top-level await again
    revised = []  # successful revisions land here
    for res in rev_results:  # walk the results
        if isinstance(res, Exception):  # this revision crashed
            log_event({"stage": "revise", "error": str(res), "loop": loop_idx})  # record it
        else:  # this revision worked
            revised.append(res)  # keep it
    print(f"Loop {loop_idx}: {len(revised)} revised answers collected.")  # how many rewrites

    # Group the revisions by question so we can swap them into the re-judge pools.
    revised_lookup = {}  # (model, question_id, rep) -> revised answer record
    for r in revised:  # walk the revisions
        revised_lookup[(r["model"], r["question_id"], r["rep"])] = r  # file by identity

    # Re-judge with the SAME number of passes as the original Stage 5 judging
    # (JUDGE_PASSES), and with the FULL field in every pool: revisers contribute
    # their fresh text, while non-revising models contribute their unchanged
    # best answers. That way "before" (3 passes against the full field) and
    # "after" (3 passes against the full field) are directly comparable.
    rejudge_jobs = []  # the re-judging work list
    for q in kept_questions:  # for every kept question
        # Assemble this question's pool from every answer we currently hold.
        pool = [  # the full field for this question
            revised_lookup.get((model, qid, rep), ans)  # fresh text if this answer was revised, else the unchanged one
            for (model, qid, rep), ans in best_answers.items()  # every answer on record
            if qid == q["id"]  # only this question's answers
        ]
        if not pool:  # nobody answered this question at all
            continue  # nothing to re-judge here
        for p in range(JUDGE_PASSES):  # same pass count as Stage 5, so deltas compare like with like
            rejudge_jobs.append(judge_pass(q, pool, pass_idx=100 * loop_idx + p))  # pass ids 100+ mark re-judges (loop 1: 100-102, loop 2: 200-202)
    rejudge_results = await asyncio.gather(*rejudge_jobs, return_exceptions=True)  # blind re-judging
    new_records = []  # the fresh scores land here
    for res in rejudge_results:  # walk the re-judge results
        if isinstance(res, Exception):  # a re-judge crashed
            log_event({"stage": "judge", "model": JUDGE_MODEL, "error": str(res), "loop": loop_idx})  # record it
        else:  # a list of fresh score records
            new_records.extend(res)  # merge them in

    # Compare: who improved after revising? Both sides are now means over
    # JUDGE_PASSES passes judged against the full field, so the delta is honest.
    new_totals = mean_total_by_model(new_records)  # model -> mean total after revision
    improved = []  # models that got better
    for model in bottom_models:  # check each reviser
        before = current_totals.get(model, 0.0)  # score before this loop
        after = new_totals.get(model, before)  # score after (fall back to before if unscored)
        delta = after - before  # the change
        print(f"  {model}: {before:.1f} -> {after:.1f}  (delta {delta:+.1f})")  # before/after row
        if delta > 0:  # the revision genuinely helped
            improved.append(model)  # remember the improvement

    # Adopt the revisions and the fresh scores as the new state of the world.
    for r in revised:  # every successful revision
        best_answers[(r["model"], r["question_id"], r["rep"])] = r  # the revision replaces the original
    # Re-judge scores go into the SEPARATE revision_records list (see the note
    # at the top of this cell) — they must never contaminate Stage 7's aggregates.
    revision_records.extend(new_records)  # keep re-judge scores for the delta report only
    for r in new_records:  # every fresh score
        latest_critique[(r["model"], r["question_id"], r["rep"])] = r["critique"]  # freshest critique wins
    current_totals.update(new_totals)  # everyone was re-judged in the full-field pool, so refresh all standings

    # Early exit: if a whole round helped nobody, further rounds are unlikely to.
    if not improved:  # no model improved this round
        print(f"Loop {loop_idx}: no model improved — stopping early.")  # say why we stop
        break  # leave the loop before wasting another round

In [ ]:
# %%
# =============================================================================
# Stage 7: Leaderboard & analysis
# =============================================================================
# Mean score per rubric dimension, overall rank, per-model variance across reps,
# latency per provider, and judge self-consistency (stdev across the 3 passes).

# Every aggregate below is computed ONLY from the original Stage 5 judging
# passes (pass ids < 100). Stage 6's re-judge records (pass ids >= 100) live in
# the separate revision_records list precisely so they cannot leak in here —
# otherwise revised answer texts would inflate the leaderboard means and make
# judge self-consistency compare two DIFFERENT texts under the same key.
original_records = [r for r in score_records if r["pass"] < 100]  # original passes only

# Per-model: average each rubric dimension across every scored record.
per_model_dims = {}  # model -> {dimension: mean score}
for r in original_records:  # walk every original-pass score record
    entry = per_model_dims.setdefault(r["model"], {d: [] for d in RUBRIC})  # model -> dimension -> list
    for d in RUBRIC:  # for each of the four dimensions
        v = r["scores"].get(d)  # this record's score on that dimension
        if isinstance(v, (int, float)):  # only real numbers count
            entry[d].append(v)  # add to the dimension's list

# Turn the per-dimension lists into a summary row per model.
rows = []  # one leaderboard row per model
for model, dims in per_model_dims.items():  # walk every scored model
    means = {d: statistics.mean(vs) if vs else 0.0 for d, vs in dims.items()}  # mean per dimension
    total = sum(means.values())  # overall score out of 40
    rows.append({"model": model, **means, "total": total})  # the row

# Rank the rows by total score, best first.
rows.sort(key=lambda r: r["total"], reverse=True)  # descending by total
for i, r in enumerate(rows, start=1):  # assign ranks
    r["rank"] = i  # 1 = best

# Name any contestants who ended up with zero scored records (e.g. all their
# calls failed), so their absence from the table is explained, not silent.
scored_models = {r["model"] for r in original_records}  # who actually got scored
unscored_models = [c["name"] for c in CONTESTANTS if c["name"] not in scored_models]  # who did not
if unscored_models:  # anyone missing from the leaderboard
    print("Note: no scored records for these models (omitted from the leaderboard): " + ", ".join(unscored_models))  # explain the gaps

# Per-model variance across reps: spread of an answer's total score between the
# REPS repetitions of the same question (are they consistent with themselves?).
rep_totals = {}  # (model, question) -> list of per-rep totals
for r in original_records:  # walk every original-pass score record
    vals = [v for v in r["scores"].values() if isinstance(v, (int, float))]  # numeric scores only
    if vals:  # only count real scores
        rep_totals.setdefault((r["model"], r["question_id"]), {})  # make room
        # Average this answer's total across passes for this rep, stored per rep.
        rep_totals[(r["model"], r["question_id"])].setdefault(r["rep"], []).append(sum(vals))  # per rep

rep_variance = {}  # model -> mean variance across its question/rep pairs
for (model, qid), per_rep in rep_totals.items():  # walk each model+question
    rep_means = [statistics.mean(ts) for ts in per_rep.values()]  # one mean total per rep
    if len(rep_means) > 1:  # need at least two reps to measure spread
        var = statistics.variance(rep_means)  # variance across the reps
        rep_variance.setdefault(model, []).append(var)  # collect per model
rep_variance = {m: statistics.mean(vs) for m, vs in rep_variance.items()}  # mean variance per model

# Latency stats: mean seconds per call, grouped by provider.
lat_by_provider = {}  # provider -> list of latencies
for a in answers:  # every original answer carries its latency
    lat_by_provider.setdefault(a["provider"], []).append(a["latency_s"])  # collect it
latency_stats = {p: statistics.mean(ls) for p, ls in lat_by_provider.items()}  # mean per provider

# Judge self-consistency: for each individual answer, how much did its total
# score wobble across the 3 judging passes? Small stdev = a steady judge.
per_answer_pass_totals = {}  # (model, question, rep) -> [total per pass]
for r in original_records:  # walk every original-pass score record (same text across passes, so this measures judge wobble only)
    vals = [v for v in r["scores"].values() if isinstance(v, (int, float))]  # numeric scores only
    if vals:  # only real scores
        key = (r["model"], r["question_id"], r["rep"])  # which answer
        per_answer_pass_totals.setdefault(key, []).append(sum(vals))  # one total per pass
consistency = [statistics.stdev(ts) for ts in per_answer_pass_totals.values() if len(ts) > 1]  # stdev per answer
mean_consistency = statistics.mean(consistency) if consistency else 0.0  # average wobble

# Show the leaderboard — pretty table if pandas is around, plain rows otherwise.
if HAVE_PANDAS:  # pretty tables available
    leaderboard_df = pd.DataFrame(rows)[["rank", "model", *RUBRIC, "total"]]  # tidy column order
    leaderboard_df["rep_variance"] = leaderboard_df["model"].map(rep_variance)  # attach variance
    display(leaderboard_df.round(2))  # show it nicely in the notebook
else:  # plain prints fallback
    for r in rows:  # one row per model
        dim_str = "  ".join(f"{d}={r[d]:.1f}" for d in RUBRIC)  # the four means
        print(f"#{r['rank']:>2} {r['model']:<40} {dim_str}  total={r['total']:.1f}  rep_var={rep_variance.get(r['model'], 0):.2f}")  # the row

# Show latency per provider and the judge's steadiness.
print("\nMean latency per provider (seconds):")  # latency section header
for p, mean_lat in sorted(latency_stats.items(), key=lambda kv: kv[1]):  # fastest first
    print(f"  {p:<12} {mean_lat:.2f}s")  # one line per provider
print(f"\nJudge self-consistency: mean per-answer stdev across passes = {mean_consistency:.2f} (lower = steadier judge)")  # the wobble number


In [ ]:
# %%
# =============================================================================
# Stage 8: Persist the final results
# =============================================================================
# The diary (arena_log.jsonl) has been appended to throughout; here we write the
# final leaderboard snapshot to arena_results.json.

# Assemble the final snapshot: leaderboard, variance, latency, consistency.
final_results = {  # everything worth keeping from the run
    "leaderboard": rows,  # ranked per-model mean scores
    "rep_variance": rep_variance,  # per-model variance across reps
    "latency_by_provider": latency_stats,  # mean latency per provider
    "judge_self_consistency_stdev": mean_consistency,  # judge steadiness
    "n_questions": len(kept_questions),  # how many questions survived the critic
    "n_contestants": len(CONTESTANTS),  # how many models competed
    "reps": REPS,  # repetitions per question
    "judge_passes": JUDGE_PASSES,  # judging passes per question
    "judge_model": JUDGE_MODEL,  # who judged
    "failures": failures,  # any fan-out failures we recorded
    "written_at": datetime.now(timezone.utc).isoformat(),  # when this snapshot was saved
}

# Write the snapshot as a single pretty-printed JSON file.
RESULTS_PATH.write_text(json.dumps(final_results, indent=2, ensure_ascii=False), encoding="utf-8")  # save the results

# Confirm where everything landed.
print(f"Diary appended to {LOG_PATH.resolve()}")  # the call-by-call log
print(f"Final results written to {RESULTS_PATH.resolve()}")  # the leaderboard snapshot